In [58]:
import gymnasium as gym
import numpy as np

In [59]:
# how frozen lake env works:
# Agent will start from state S, its goal is to reach state G
# Its a 4*4 grid so state s: 0,1, .., 15, containing potholes (termination states)
# Agent action a: 0(left), 1(down), 2(right), 3(up)

env = gym.make("FrozenLake-v1", render_mode="human")
observation, info = env.reset()

episode_over = False
while not episode_over:
    action = env.action_space.sample()  # agent policy that uses the observation and info
    observation, reward, terminated, truncated, info = env.step(action)

    episode_over = terminated or truncated

env.close()


In [60]:
# we can render in different mode(text base mode)

# is_slippery flag: even if we take action of going down, we might end up
# in a state other then directly below one(left or right not up)! reason to model real world!
env = gym.make("FrozenLake-v1", render_mode="ansi", is_slippery = True) 
env = env.unwrapped         # because it was wrapped in time counter to give truncated error
observation, info = env.reset()

print(f"previous state: {env.render()}")

action = env.action_space.sample()
state, reward, terminated, truncated, info = env.step(action)

print(f"current state: {env.render()}")

print(f"Agent is in state: {state}, received reward: {reward}, is episode terminated: {terminated}")


previous state: 
SFFF
FHFH
FFFH
HFFG

current state:   (Down)
SFFF
FHFH
FFFH
HFFG

Agent is in state: 1, received reward: 0, is episode terminated: False


In [61]:
# this env will give you the transition matrix
# In real-world we can't get the transition matrix
env.P[1][1]

# here the matrix env.P is
# state, action, transition prob, next state, reward, Termination 

[(0.33333333333333337, 0, 0, False),
 (0.3333333333333333, 5, 0, True),
 (0.33333333333333337, 2, 0, False)]

In [62]:
# Value Iteration
# one step: Bellman optimality backup V(s) = max_a sum_s' p(s'|s,a) * [r + gamma * V(s')]

num_states = env.observation_space.n
num_actions = env.action_space.n
gamma = 0.9
epoch = 20

state_value = np.zeros(num_states)
policy = np.zeros(num_states)

for i in range(epoch):
    for state in range(num_states):
        action_value = np.zeros(num_actions)
        for action in range(num_actions):
            # P[state][action] will give info about possible next state which could be one of three
            # one-step lookahead: action value Q(s,a)
            for TransProb, nextstate, reward, terminated in env.P[state][action]:
                action_value[action] += TransProb * (reward + gamma * state_value[nextstate])

        # take the best action value as the new state value
        state_value[state] = max(action_value)
        policy[state] = np.argmax(action_value)
    print("*** after epoch ", i+1,"***")
    print("the state value \n", state_value.reshape(4,4))
    print("the best policy \n", policy.reshape(4,4))

*** after epoch  1 ***
the state value 
 [[0.         0.         0.         0.        ]
 [0.         0.         0.         0.        ]
 [0.         0.         0.         0.        ]
 [0.         0.         0.33333333 0.        ]]
the best policy 
 [[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 1. 0.]]
*** after epoch  2 ***
the state value 
 [[0.         0.         0.         0.        ]
 [0.         0.         0.         0.        ]
 [0.         0.         0.1        0.        ]
 [0.         0.1        0.46333333 0.        ]]
the best policy 
 [[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 1. 1. 0.]]
*** after epoch  3 ***
the state value 
 [[0.         0.         0.         0.        ]
 [0.         0.         0.03       0.        ]
 [0.         0.06       0.166      0.        ]
 [0.         0.187      0.52843333 0.        ]]
the best policy 
 [[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 2. 1. 0.]]
*** after epoch  4 ***
the state value 
 [[0.         0.         0.009   

Please read this to see the optimal policy:

<img src="optimul%20solutions.jpeg" alt="optimal policy" width="600">

In [63]:
env = gym.make("FrozenLake-v1", render_mode= None , is_slippery = True)

print("Evaluating learned policy...")
total_rewards = 0
num_eval_episodes = 1000

for _ in range(num_eval_episodes):
    state, info = env.reset()
    terminated = False
    truncated = False
    episode_reward = 0
    while (not terminated and not truncated):
        # For evaluation, use a greedy policy (no exploration)
        action = policy[state]
        next_state, reward, terminated, truncated, info = env.step(action)
        episode_reward += reward
        state = next_state

    total_rewards += episode_reward

print(f"Average reward over {num_eval_episodes} evaluation episodes: {total_rewards / num_eval_episodes:.4f}")

Evaluating learned policy...
Average reward over 1000 evaluation episodes: 0.7470
